In [1]:
using Distributed
using JLD2
using Plots

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end ;
@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
    using BlockDiagonals
end

      From worker 59:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 59:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 39:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 39:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 47:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 47:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 77:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 77:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 90:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file i

      From worker 63:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 63:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 72:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 72:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 95:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 95:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 86:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 86:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 92:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file i

      From worker 22:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 22:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 37:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 37:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 18:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 18:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 15:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file if no other process is resolving.
      From worker 15:	└   lock_file = "/local/home/maolinml/latticealgorithms.jl/.CondaPkg/lock"
      From worker 50:	┌ Info: CondaPkg: Waiting for lock to be freed. You may delete this file i

In [2]:
println("num_cores = $(num_cores)")

num_cores = 96


In [21]:
type_lattice = "surface_square"
num_super_samples = 1
num_samples = 1042 * 96
Kmax = 100
Nv = 3

dmin, dmax = 7, 7
drange = dmin : 2 : dmax

σrange = [0.606]

σdrange = []
for σ in σrange
    for d in drange
        push!(σdrange, [σ, d])
    end
end
println(length(σdrange))

num_samples_each_core = Int(ceil(num_samples/num_cores))
num_samples = Int(num_samples_each_core * num_cores);
num_total_samples = num_super_samples * num_samples
println([num_samples_each_core, num_samples, num_total_samples])



logfile = "$(type_lattice)_mwms_$(dmin)_$(dmax)_$(min(σrange...))_$(max(σrange...))_$(Kmax)_$(Nv)_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

1
[1042, 100032, 100032]


In [ ]:
p_mld_list2 = Dict(σdrange.=>[[[0.0, 0.0, 0.0, 0.0] for _ in 1 : Kmax] for _ in 1 : length(σdrange)])
t_mld_list2 = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])

total_t = @elapsed for ind in 1 : num_super_samples  
    @time results = pmap(1:num_cores) do _        
        p_mld_list = Dict(σdrange.=>[[] for _ in 1 : length(σdrange)])
        t_mld_list = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])
        
        for (ind_σd, σd) in enumerate(σdrange)
            σ, d = σd[1], Int(σd[2])
            
            S_hex = [2 1; 0 √3] / (12)^(1/4)
            S_hex_T = Matrix(transpose(S_hex))

            M0 = surface_code_M(d)
            M = M0 * BlockDiagonal([S_hex_T for _ in 1 : d^2])

            Mperp = GKP_logical_operator_generator(M)
            Ω = Ω_matrix(M)

            X_logical = surface_code_X_logicals(d)[1]
            X = zeros(2d^2)
            X[2 .* X_logical .- 1] .= 1
            Z_logical = surface_code_Z_logicals(d)[1]
            Z = zeros(2d^2)
            Z[2 .* Z_logical] .= 1

            Z = BlockDiagonal([S_hex for _ in 1:d^2]) * Z .* √π
            X = BlockDiagonal([S_hex for _ in 1:d^2]) * X .* √π            
            
            p_mld = [[0.0, 0.0, 0.0, 0.0] for _ in 1 : Kmax]
            t_mld = 0       

            σdtime = @elapsed for _ in 1 : num_samples_each_core
                ξ = σ * randn(2d^2)
                ξ2 = √(2π) * M * inv(Ω) * ξ
                s = ξ2 - floor.(ξ2/(2π)) * 2π
                
                t_mld += @elapsed begin
                    ηs = -transpose(Ω*Mperp) * s/√(2π) ;                     
                    ηsq = ηs[1:2:end]

                    ps_I, ps_X, ps_Y, ps_Z = mwms_surface_hexagonal(ηs, σ, Kmax; Nv=Nv) 
                    
                    for k in 1 : Kmax
                        p_I, p_X, p_Y, p_Z = ps_I[k], ps_X[k], ps_Y[k], ps_Z[k]
                        prob = max(p_I, p_X, p_Y, p_Z)
                        if prob == p_I
                            lstar = zeros(length(ηs))
                        elseif prob == p_X
                            lstar = X
                        elseif prob == p_Z
                            lstar = Z
                        elseif prob == p_Y
                            lstar = X+Z
                        end                        
                        
                        rec = -ηs + lstar
                        final_error = (rec+ξ)

                        nx = abs(transpose(final_error) * Ω * Z / π)
                        nz = abs(transpose(final_error) * Ω * X / π)
                        nx = mod(round(nx), 2)
                        nz = mod(round(nz), 2)

                        @assert abs(round(Int, nx)-nx)<1e-5
                        @assert abs(round(Int, nz)-nz)<1e-5

                        if nx ≈ 0 && nz ≈ 0
                            fidelity = [1, 0, 0, 0]
                        elseif nx ≈ 0 && nz ≈ 1
                            fidelity = [0, 0, 1, 0]
                        elseif nx ≈ 1 && nz ≈ 0
                            fidelity = [0, 1, 0, 0]
                        elseif nx ≈ 1 && nz ≈ 1
                            fidelity = [0, 0, 0, 1]
                        end
                        
                        p_mld[k] .+= fidelity
                    end
                end  
            end
            p_mld_list[[σ, d]] = p_mld
            t_mld_list[[σ, d]] += t_mld
            
            if myid() == 2
                # Print the progress of the 2nd worker
                println(["$(ind)/$(num_super_samples), $(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))"])
                open(logfile, "a") do file
                    write(file, "$(ind)/$(num_super_samples), $(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))\n")
                end
            end
        end
        return p_mld_list, t_mld_list
    end ;     
    p_mld_list  = merge(.+, [res[1] for res in results]...)
    t_mld_list  = merge(+, [res[2] for res in results]...)
    
    p_mld_list2 = merge(.+, p_mld_list2, p_mld_list)
    t_mld_list2 = merge(+, t_mld_list2, t_mld_list)
    
end

println(total_t)
map!(x->x./num_total_samples, values(p_mld_list2))
map!(x->x./num_total_samples, values(t_mld_list2))

fn = "$(type_lattice)_mwms_$(dmin)_$(dmax)_$(min(σrange...))_$(max(σrange...))_$(Kmax)_$(Nv)_$(num_total_samples).jld2"
jldsave(fn; 
    σrange=σrange, 
    drange=drange, 
    num_samples=num_total_samples,
    p_list = p_mld_list2,
    t_list = t_mld_list2,        
)    


In [ ]:
sort(load(fn))

# Check with existing data

In [ ]:
using JLD2
using Plots

In [ ]:
function get_p0list_sorted(p_list, drange, σrange)
    p0list_sorted = sort(p_list)
    p0list_sorted = collect(values(p0list_sorted))
    p0list_sorted = reshape(p0list_sorted, (length(drange), length(σrange)))
    p0list_sorted = [p0list_sorted[:,i] for i in 1:size(p0list_sorted,2)]
    return p0list_sorted
end

In [ ]:
new_data = sort(load(fn))
new_p_list = new_data["p_list"]
new_p_list_sorted = get_p0list_sorted(new_p_list, drange, σrange)

In [ ]:
old_data = sort(load("data/surface_hexagonal_mwms_7_7_0.596_0.607_500_3_1000320.jld2"))
old_p_list = Dict()
for (k, v) in old_data["p_list"]
    if k[2] ∈ drange && k[1] ∈ σrange
        old_p_list[k] = v
    end
end

old_p_list_sorted = get_p0list_sorted(old_p_list, drange, σrange)

In [ ]:
linecolors = get_color_palette(:auto, plot_color(:white))
linecolorind = 0    

g = plot()
for (ind_d, d) in enumerate(drange)
    for (ind_σ, σ) in enumerate(σrange)
        linecolorind +=1
        new_ps = new_p_list_sorted[ind_σ][ind_d]
        new_ps = [item[1] for item in new_ps]
        old_ps = old_p_list_sorted[ind_σ][ind_d]
        old_ps = [item[1] for item in old_ps]        
        yerrnew = sqrt.(new_ps .* (1 .- new_ps) ./ num_total_samples)
        yerrold = sqrt.(old_ps .* (1 .- old_ps) ./ num_total_samples)
        plot!(new_ps, label="new data, d=$d, σ=$σ", marker=:circle, color=linecolors[linecolorind], yerr=1.5yerrnew)
        plot!(old_ps, label="old data, d=$d, σ=$σ", marker=:star, color=linecolors[linecolorind], yerr=1.5yerrold)
    end
end
plot!(xlabel="K", ylabel="fidelity", size=(1200, 400))